# Molecular dynamics analysis of HIV-1 protease

This Jupyter notebook presents a molecular dynamics analysis of the Human Immunodeficiency Virus type I (HIV-1) protease. We simulated HIV-1 protease (PDB ID: 1HVR) without its ligand for 201 ns. Then, a total of 201 frames were extracted at regular intervals of 1 ns from the molecular dynamics’ trajectory.

Here, we describe the conformational changes of a cavity that defines the active site of the HIV-1 protease, which is an effective therapeutic target. The HIV-1 protease catalytic cycle involves movements of β -hairpins, called 'flaps', which control the accessibility of substrates to the active site of the homodimer. Further, we performed a occurence of cavity points, that were detected in at least two frames, and we plotted all properties (volume, area, depth and hydropathy) throughout the simulation.

In [1]:
# Import required modules
import os
import numpy
import pickle
import pyKVFinder
import KVFinderMD
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics import silhouette_score
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import linkage
import seaborn as sns

## HIV-1 protease trajectory

## Cavity detection and characterization workflow


In [ ]:
%%timeit -r 3 -n 1
# Create KVFinderMD object
md = KVFinderMD.KVFinderMD()

# Load HIV-1 protease trajectory
md.read_trajectory("data/HIV.pdb")

# Custom detection parameters
probe_out = 12.0
volume_cutoff = 50.0

# Perform detection and characterization
md.detect(
    # Analysis modes
    analyze_constitutional=True, 
    export_occurrence=True,
    analyze_spatial=True, 
    analyze_depth=True, 
    analyze_hydropathy=True,
    # Custom parameters
    probe_out=probe_out, 
    volume_cutoff=volume_cutoff,
    # Miscellaneous
    basedir='results/spde',
    verbose=True
)

# Write characterization to file
md.write('results/spde/results.toml')

# Save md with pickle
with open("results/spde/md.pkl", "wb") as f:
    pickle.dump(md, f)

## Cavity alignment analysis

We explored three formulations for structural alignment of cavities:
- 3D grid alignment;
- (2D) Contact matrix alignment;
- (2D) Distance matrix alignment - Inspired on DALI structural alignment.


In [2]:
# Create directories
os.makedirs("results/spde/grid", exist_ok=True)
os.makedirs("results/spde/contact", exist_ok=True)
os.makedirs("results/spde/distance", exist_ok=True)

### 3D grid alignment: clustering 3D grid of each cavity detected throughout the molecular dynamics simulation

Here, we separate each cavity in a different boolean grid (1: cavity; 0: everything else).

In [3]:
if os.path.exists("results/spde/md.pkl"):
    with open("results/spde/md.pkl", "rb") as f:
        md = pickle.load(f)
    md.occurrence._gap = 1

In [ ]:
# Separate each cavity in a different boolean grid
cavities = list()
frames = list()

for n in range(md.n_frames):
    f = md.frame(n)
    frames.append(n)
    
    for ncav in range(f.n_cavities):
        cavities.append(f.cavities == ncav+2)
        frames.append(f)

cavities = numpy.asarray(cavities)
frames = numpy.asarray(frames)

# Show grids
print(cavities.shape)

In [ ]:
# Prepare data
# NOTE: Memory consuming step
cavities = cavities.astype(bool).reshape(672, -1)

# Show data
print(cavities.shape)

In [ ]:
X = cavities

D = squareform(pdist(X, metric="correlation"))

Z = linkage(pdist(X), method="complete")

sns.set_theme(context="poster", style="white")

g = sns.clustermap(
    D,
    row_linkage=Z,
    col_linkage=Z,
    cmap="mako",
    figsize=(20, 20),
    xticklabels=False,
    yticklabels=False,
    tree_kws={"linewidths": 1.5},
    dendrogram_ratio=(0.18, 0.18),
    cbar_pos=(0.05, 0.85, 0.05, 0.18),
)

# Colorbar
cbar = g.ax_heatmap.collections[0].colorbar
cbar.set_ticks([-0.99, 0.99])
cbar.set_ticklabels(["Similar", "Distinct"], size=12)
cbar.set_label(
    "Dissimilarity",
    fontsize=22,
    labelpad=-150,
)
 
# Transparent background
g.fig.patch.set_alpha(0)
 
plt.savefig(
    "results/spde/grid/clustermap.png",
    dpi=600,
    bbox_inches="tight",
    transparent=True,
)

#### Agglomerative clustering based on Silhouette Scores

In [ ]:
%%timeit -r 3 -n 1
preds = AgglomerativeClustering(n_clusters=10, metric='correlation', linkage="complete").fit_predict(cavities)

In [ ]:
preds = AgglomerativeClustering(n_clusters=10, metric='correlation', linkage="complete").fit_predict(cavities)

KVFinderMD.silhouette(
    cavities,
    preds,
    metric='correlation',
    filename=f'results/spde/grid/silhouette.png'
)

# Write cavities to file
for i, cav in enumerate(cavities):
    cav = cav.reshape(160, 126, 105)
    B = numpy.ones(cav.shape) * preds[i]
    pyKVFinder.export(
        f'results/spde/grid/{preds[i]}/cavity-{i:03d}.pdb', 
        cav.astype(int) * 2, 
        None,
        md.kvtraj._vertices,
        md.kvtraj._step,
        B=B
    )

### (2D) Contact matrices alignment: clustering contact matrix of each cavity detected throughout the molecular dynamics simulation

Here, we define a contact matrix for each cavity, considering the interface residues surrounding it.

#### Explore distance metrics in 2D comparison of contact matrices

The distance metrics are used to assess the similarity of the adjacency matrices, ie the detected cavities.

The standard metrics: 
- Euclidean distance
- Correlation

Pairwise distances for booleans:
- Dice dissimilarity
- Hamming distance
- Jaccard dissimilarity
- Rogers-Tanimoto dissimilarity
- Russell-Rao dissimilarity
- Sokal-Michener dissimilarity
- Sokal-Sneath dissimilarity
- Yule dissimilarity

Discussion about those metrics:
- https://www.ibm.com/docs/en/spss-statistics/SaaS?topic=measures-distances-similarity-binary-data
- https://stats.stackexchange.com/questions/61705/similarity-coefficients-for-binary-data-why-choose-jaccard-over-russell-and-rao

In [ ]:
# Create CavityAlignment object
alignment = KVFinderMD.CavityAlignment(md)

In [ ]:
# Explore contact matrix alignment
alignment.explore(method="contact")

In [ ]:
alignment.scores

In [ ]:
%%timeit -r 3 -n 1
# Alignment
alignment.align(method='contact', affinity="dice")

In [ ]:
alignment.align(method='contact', affinity="dice")

# Plot silhouette
KVFinderMD.silhouette(
    alignment.contacts.reshape(672, -1),
    alignment.clusters,
    metric='dice',
    filename=f'results/spde/contact/silhouette.png'
)

# Write cavities to file
for i, cav in enumerate(cavities):
    cav = cav.reshape(160, 126, 105)
    B = numpy.ones(cav.shape) * alignment.clusters[i]
    pyKVFinder.export(
        f'results/spde/contact/{alignment.clusters[i]}/cavity-{i:03d}.pdb', 
        cav.astype(int) * 2, 
        None, 
        md.kvtraj._vertices, 
        md.kvtraj._step,
        B=B
    )

In [ ]:
X = alignment.contacts.reshape(672, -1)

D = squareform(pdist(X, metric="dice"))

Z = linkage(pdist(X), method="complete")

sns.set_theme(context="poster", style="white")

g = sns.clustermap(
    D,
    row_linkage=Z,
    col_linkage=Z,
    cmap="mako",
    figsize=(20, 20),
    xticklabels=False,
    yticklabels=False,
    tree_kws={"linewidths": 1.5},
    dendrogram_ratio=(0.18, 0.18),
    cbar_pos=(0.05, 0.85, 0.05, 0.18),
)

# Colorbar
cbar = g.ax_heatmap.collections[0].colorbar
cbar.set_ticks([-0.99, 0.99])
cbar.set_ticklabels(["Similar", "Distinct"], size=12)
cbar.set_label(
    "Dissimilarity",
    fontsize=22,
    labelpad=-150,
)
 
# Transparent background
g.fig.patch.set_alpha(0)
 
plt.savefig(
    "results/spde/contact/clustermap.png",
    dpi=600,
    bbox_inches="tight",
    transparent=True,
)

### (2D) Distance matrices alignment: clustering contact matrix of each cavity detected throughout the molecular dynamics simulation

In [ ]:
# Explore contact matrix alignment
alignment.explore(method="distance")

In [ ]:
alignment.scores

In [ ]:
%%timeit -r 5 -n 3
# Alignment
alignment.align(method='distance', affinity="cosine")

In [ ]:
alignment.align(method='distance', affinity="cosine")

# Plot silhouette
KVFinderMD.silhouette(
    alignment.distances.reshape(672, -1),
    alignment.clusters,
    metric="cosine",
    filename=f'results/spde/distance/silhouette.png'
)

# Write cavities to file
for i, cav in enumerate(cavities):
    cav = cav.reshape(160, 126, 105)
    B = numpy.ones(cav.shape) * alignment.clusters[i]
    pyKVFinder.export(
        f'results/spde/distance/{alignment.clusters[i]}/cavity-{i:03d}.pdb', 
        cav.astype(int) * 2, 
        None, 
        md.kvtraj._vertices, 
        md.kvtraj._step,
        B=B
    )

In [ ]:
X = alignment.distances.reshape(672, -1)

D = squareform(pdist(X, metric="cosine"))

Z = linkage(pdist(X), method="complete")

sns.set_theme(context="poster", style="white")

g = sns.clustermap(
    D,
    row_linkage=Z,
    col_linkage=Z,
    cmap="mako",
    figsize=(20, 20),
    xticklabels=False,
    yticklabels=False,
    tree_kws={"linewidths": 1.5},
    dendrogram_ratio=(0.18, 0.18),
    cbar_pos=(0.05, 0.85, 0.05, 0.18),
)

# Colorbar
cbar = g.ax_heatmap.collections[0].colorbar
cbar.set_ticks([0.01, 0.99])
cbar.set_ticklabels(["Similar", "Distinct"], size=12)
cbar.set_label(
    "Dissimilarity",
    fontsize=22,
    labelpad=-150,
)
 
# Transparent background
g.fig.patch.set_alpha(0)
 
plt.savefig(
    "results/spde/distances/clustermap.png",
    dpi=600,
    bbox_inches="tight",
    transparent=True,
)